# Gerar Legenda MULTICOR — 5 Idiomas (classificação gramatical por palavra)

> 🖥️ **CPU basta** — este notebook não usa GPU. Deixe o acelerador em *Nenhum*: não muda nada aqui e poupa sua cota de GPU, que é limitada.

Gera o arquivo `.ass` com a legenda colorida — não queima em nenhum vídeo
ainda (isso é o notebook `caption-multicolor-burn.ipynb`, separado de
propósito, pra dar espaço pra correção manual no meio do caminho).

No final, duas ações **separadas**: baixar o `.ass` pra revisar/corrigir,
e (só depois de confirmar que ficou bom) salvar no Drive.

## Quatro camadas, cada uma refazível sozinha

O Stanza e o Kiwi erram, e o erro deles é mudo: sai uma cor plausível e
ninguém percebe até assistir o vídeo. A defesa é separar o que vem de onde.

```
BRUTO            o que o analisador disse   →  <nome>_analise_bruta_<lang>.json
  ↓
MAPEAMENTO       o que a etiqueta significa →  classificacao.py
  ↓
CENTRAL          onde a etiqueta engana     →  dados_lexico/classes-correcoes.json
  ↓
SUA CORREÇÃO     o que só você sabe         →  a coluna `classe` do CSV
  ↓  .ass
```

**Célula 4 — o bruto e as regras.** Guarda a análise crua (token com as
palavras sintáticas dentro, lema, upos e traços) e aplica a central de
correções automáticas. Se o bruto já existir e bater com a legenda, ela
**nem roda o analisador**: remapeia em segundos. E remapear não apaga o que
você corrigiu à mão — a peça guarda `classe_automatica`, e onde ela difere
de `classe`, a diferença é humana.

**Célula 5 — revisar.** Aponta ONDE olhar, em vez de pedir que você leia
3.300 peças. No Mateus 2 são 39 apontamentos (1,2%). Saem dois arquivos:

| arquivo | pra quê |
|---|---|
| `<nome>_classes_revisar.html` | **achar** o erro — a legenda pintada com as cores de verdade, os 5 idiomas empilhados como no vídeo |
| `<nome>_classes_revisar.csv` | **corrigir** — abre no Sheets, você muda a coluna `classe` |

**Célula 6 — aplicar.** Lê o CSV de volta, recusa classe que não existe, e
sugere quais das suas correções merecem virar regra na central (só as que
você repetiu — correção de uma vez só costuma ser contexto). Do CSV só a
coluna `classe` atravessa: `lema`, `feats` e `classe_automatica` ficam de
fora da planilha de propósito, e são preservados da memória.

Se a revisão não achou nada, pule a 6 e vá direto pra 7.


In [5]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  1. SETUP                                                        ║
# ╚══════════════════════════════════════════════════════════════════╝
!pip install -q stanza kiwipiepy

import shutil, sys
from pathlib import Path

import stanza
from kiwipiepy import Kiwi
from google.colab import drive

try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)

PASTA_DRIVE_RAIZ_MODULOS = "narrated_video"
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ_MODULOS}/pipeline/modulos")
DESTINO = Path("/content/pipeline")
if PASTA_MODULOS.exists():
    if DESTINO.exists():
        shutil.rmtree(DESTINO)
    shutil.copytree(PASTA_MODULOS, DESTINO)
    print(f"✅ {len(list(DESTINO.glob('*.py')))} módulos copiados")

    # ── A cópia trouxe TODOS os módulos? ───────────────────────────────────────
    # "N módulos copiados" sozinho não quer dizer nada. E o modo de falhar aqui é
    # traiçoeiro: o Drive montado do Colab popula a listagem da pasta com atraso,
    # então um copytree logo depois do mount às vezes enxerga só parte dos
    # arquivos. Já aconteceu de copiar 13 de 31 -- com visto verde -- e o notebook
    # quebrar muito depois, num import, longe da causa.
    #
    # A conferência é de três pontas, porque a causa muda o conserto:
    #   manifesto  o que o repositório tem  (versionado; chega pela cópia)
    #   Drive      o que chegou lá
    #   VM         o que a cópia desta célula trouxe
    # A conferência tem duas perguntas, e SÓ UMA delas precisa do manifesto:
    #
    #   Drive → VM   a cópia acima trouxe tudo?      dá pra ver aqui mesmo
    #   repo → Drive o Drive está em dia?            só o manifesto sabe
    #
    # A versão anterior amarrava as duas ao manifesto: sem ele, imprimia um
    # aviso e seguia SEM CONFERIR NADA. Foi assim que "✅ 13 modules copied"
    # passou com visto verde num Drive que tinha 31 -- justamente no dia em
    # que o manifesto ainda não existia. Comparar 13 com 31 nunca dependeu de
    # manifesto nenhum.
    _no_drive = {f.name for f in PASTA_MODULOS.glob("*.py")}
    _na_vm    = {f.name for f in DESTINO.glob("*.py")}

    # ── Drive → VM ────────────────────────────────────────────────────────
    # O Drive montado do Colab popula a listagem da pasta com atraso, então um
    # copytree logo depois do mount às vezes enxerga só parte dos arquivos.
    # Uma segunda passada, com o mount já quente, costuma resolver.
    _nao_copiados = sorted(_no_drive - _na_vm)
    if _nao_copiados:
        print(f"   ⏳ {len(_nao_copiados)} módulo(s) não vieram na 1ª passada — copiando de novo")
        for _n in _nao_copiados:
            shutil.copyfile(PASTA_MODULOS / _n, DESTINO / _n)
        _na_vm = {f.name for f in DESTINO.glob("*.py")}
        _nao_copiados = sorted(_no_drive - _na_vm)
    if _nao_copiados:
        print(f"\n🚨 {len(_nao_copiados)} módulo(s) estão no Drive mas não copiaram:")
        for _n in _nao_copiados:
            print(f"     {_n}")
        raise SystemExit("Rode ESTA célula de novo — o Drive montado ainda estava acordando.")
    print(f"   ✅ os {len(_no_drive)} módulos do Drive chegaram na VM")

    # ── repositório → Drive ───────────────────────────────────────────────
    _manifesto = PASTA_MODULOS / "_manifesto.txt"
    if not _manifesto.exists():
        print("   ⚠️  sem _manifesto.txt: não dá pra saber se o DRIVE está atrás")
        print("      do repositório. Ele é versionado — rode o repositorio-sincronizar.")
    else:
        _esperados = {l.strip() for l in _manifesto.read_text().splitlines()
                      if l.strip() and not l.startswith("#")}
        _fora_do_drive = sorted(_esperados - _no_drive)
        if _fora_do_drive:
            print(f"\n🚨 {len(_fora_do_drive)} módulo(s) não estão no DRIVE:")
            for _n in _fora_do_drive:
                print(f"     {_n}")
            raise SystemExit("Rode o repositorio-sincronizar.ipynb — o Drive está atrás do repositório.")
        print(f"   ✅ e batem com os {len(_esperados)} do manifesto")

    # ── O Python está segurando a versão anterior? ────────────────────────
    # Copiar arquivo novo por cima não desfaz um import já feito: o Python
    # guarda o módulo em sys.modules e reaproveita. Numa sessão longa, isso
    # faz o notebook rodar com o config.py de ontem mesmo depois de um sync
    # perfeito -- e o sintoma aparece longe da causa (nome de arquivo que
    # mudou, padrão que era pra ter mudado e não mudou). Descarregar aqui
    # equivale a reiniciar o runtime, sem perder o resto da sessão.
    _recarregar = [_n for _n, _m in list(sys.modules.items())
                   if getattr(_m, "__file__", None) and str(DESTINO) in str(_m.__file__)]
    for _n in _recarregar:
        del sys.modules[_n]
    if _recarregar:
        print(f"   ♻️  {len(_recarregar)} módulo(s) já importados foram descarregados —")
        print(f"      o import vai reler a cópia nova (rode as células seguintes de novo)")
else:
    print(f"❌ Pasta de módulos não encontrada: {PASTA_MODULOS}")
if str(DESTINO) not in sys.path:
    sys.path.insert(0, str(DESTINO))

# ── O ambiente combina com o que este notebook faz? ────────────────────────
# Cota de GPU do Colab é limitada e some sem aviso -- e parte da nossa foi
# gasta em notebook que não usa GPU pra nada, rodando com GPU só porque a
# seleção ficou de antes. Silencioso quando combina.
try:
    from ambiente import avisar_gpu
    avisar_gpu(precisa=False)
except Exception:
    pass

# ⚠️ Os módulos abaixo (classificacao.py, classificacao_ko.py, cores.py,
# renderizacao.py) precisam estar na MESMA pasta
# narrated_video/pipeline/modulos/ do Drive, junto com os antigos, pra esse
# copytree acima já trazer eles também. Se der erro de import na célula 4,
# é sinal de que faltou subir algum desses 4 arquivos pro Drive.

kiwi = Kiwi()
print("✅ Stanza e Kiwi prontos")


Mounted at /content/drive
✅ 18 módulos copiados
✅ Stanza e Kiwi prontos


In [6]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  2. CONFIGURAÇÃO                                                 ║
# ╚══════════════════════════════════════════════════════════════════╝
NOME_ORACAO = "40_Matt_02"
PASTA_DRIVE_RAIZ = "narrated_video"

IDIOMA_MESTRE = "en"  # idioma de referência — usa o SRT "whisper" (bruto, não
                      # sincronizado); os outros idiomas usam o conjunto já
                      # sincronizado pelo tempo do mestre (sem sufixo)

IDIOMAS_STANZA = {"pt": "pt", "en": "en", "es": "es", "fr": "fr"}  # idiomas que usam Stanza
IDIOMA_KIWI = "ko"  # idioma que usa Kiwi (só coreano, por enquanto)

BOX_BORDER = 6  # espessura da caixa colorida, em px

# Separador entre peças da MESMA palavra escrita — só o coreano tem isso, onde
# um morfema não pode ser separado do seguinte por um espaço normal sem quebrar
# a palavra. Com nada entre eles, as bordas de 6px encostam e as sílabas saem
# espremidas. "\u2009" é o espaço fino; "\u200a" é ainda mais estreito, e ""
# volta ao comportamento antigo (colado).
ESPACO_ENTRE_PECAS_COLADAS = "\u2009"

# ── Margem lateral e siglas de idioma ───────────────────────────────────────
# MARGEM_LATERAL reserva px dos DOIS lados. A da esquerda é onde ficam as
# siglas (PT/EN/ES/FR/KO/ZH), na altura de cada linha; a da direita existe pra
# a legenda centralizada não ficar torta.
#
# As siglas ficam DE LADO, não acima da linha: são informação de consulta --
# você olha uma vez e sabe qual linha é a sua --, e no meio do texto
# disputariam atenção com a leitura a cada bloco.
#
# Linha que não couber na largura restante é ENCOLHIDA pra caber (o ASS não
# tem isso nativo; a conta é feita no renderizacao.py). Medido no Mateus 2:
# com 70px, 17 das 215 linhas encolhem, e a mediana delas encolhe 5%. Sem
# margem nenhuma UMA linha já transbordava a tela -- o francês do bloco 31,
# com 1403px numa tela de 1280.
#
# 0 tira a margem, mas NÃO o encolhimento: linha mais larga que a TELA
# continua sendo reduzida — deixar texto sair da tela nunca é o que se quer.
MARGEM_LATERAL = 70

# ── AS CAMADAS DESTE VÍDEO ──────────────────────────────────────────────────
# Declaração ÚNICA: mora no Drive (`<nome>_camadas.json`), junto do vídeo, e
# TODOS os notebooks de legenda a leem e obedecem. A decisão é do vídeo, não de
# quem está rodando -- antes eram quatro interruptores em três notebooks, e
# declarar num e esquecer no outro dava vídeo sem a camada.
#
#   None  = usa o que já está declarado (ou o padrão, na primeira vez)
#   dict  = REDECLARA, e os outros notebooks passam a obedecer a nova
#
# Camada ligada cujo arquivo não existe vira ERRO na queima, não aviso: quem
# ligou espera ver, e aviso no meio de saída longa passa batido.
CAMADAS = None
# CAMADAS = {"siglas_idioma": True, "indicador_versiculo": True, "titulo_trecho": True}

print(f"Vídeo: {NOME_ORACAO}")


Vídeo: 40_Matt_02


In [7]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  3. BAIXAR OS SRTs — o mestre usa a LEGENDA MESTRE, os outros o     ║
# ║  conjunto JÁ SINCRONIZADO (sem sufixo, gerado pelo                ║
# ║  caption-multilang-generate.ipynb) — não usa mais o "whisper" bruto ║
# ║  dos outros idiomas, que tem timing próprio de cada dublagem e    ║
# ║  fica fora de sincronia por conteúdo.                             ║
# ╚══════════════════════════════════════════════════════════════════╗
from config import PipelineConfig
from drive_utils import DriveClient
from srt_utils import escolher_legenda_mestre, ler_srt

drive_client = DriveClient.get()
config = PipelineConfig(NOME_ORACAO=NOME_ORACAO, PASTA_DRIVE_RAIZ=PASTA_DRIVE_RAIZ,
                        IDIOMA_MESTRE=IDIOMA_MESTRE)

# ── camadas: uma declaração só, obedecida por todos os notebooks ───────────
import camadas as _cm
_arq_camadas = Path(config.nome_camadas)
drive_client.download(config.pasta_oracao, _arq_camadas.name, _arq_camadas)
CAMADAS_ATIVAS, _mudou = _cm.resolver(_arq_camadas, CAMADAS)
if _mudou:
    _cm.salvar(CAMADAS_ATIVAS, _arq_camadas)
    drive_client.upload(_arq_camadas, config.pasta_oracao, "application/json")
print(_cm.descrever(CAMADAS_ATIVAS, "declarado aqui" if CAMADAS else "do Drive"))

legendas_por_idioma_raw = {}

todos_idiomas = list(IDIOMAS_STANZA.keys()) + [IDIOMA_KIWI]
for idioma in todos_idiomas:
    config = PipelineConfig(NOME_ORACAO=NOME_ORACAO, PASTA_DRIVE_RAIZ=PASTA_DRIVE_RAIZ, IDIOMA_MESTRE=IDIOMA_MESTRE)

    # mestre = a LEGENDA MESTRE (`<nome>_mestre.srt`), que é a própria
    # referência e não passa por sincronização; os outros = sem sufixo (o
    # conjunto já sincronizado pelo tempo do mestre).
    #
    # Antes isto lia o `_whisper_<mestre>.srt` direto, o que era um segundo
    # lugar decidindo quem é a mestre -- e que discordaria do resto do
    # pipeline no dia em que a mestre fosse outro arquivo.
    if idioma == IDIOMA_MESTRE:
        nome_arquivo, _aviso = escolher_legenda_mestre(
            config.nome_legenda_mestre, config.nomes_legenda_mestre_legado,
            lambda n: drive_client.download(config.pasta_oracao, n, Path(n))
                      and Path(n).exists())
        if _aviso:
            print(f"  ⚠️  {_aviso}")
    else:
        nome_arquivo = config.nome_srt(idioma)

    destino_local = Path(nome_arquivo)
    ok = destino_local.exists() or drive_client.download(
        config.pasta_oracao, nome_arquivo, destino_local)
    if not ok:
        print(f"  ⚠️  {idioma.upper()}: não achei '{nome_arquivo}' — pulando")
        continue
    legendas_por_idioma_raw[idioma] = ler_srt(destino_local)
    print(f"  ✅ {idioma.upper()} ({nome_arquivo}): {len(legendas_por_idioma_raw[idioma])} bloco(s)")

if not legendas_por_idioma_raw:
    raise FileNotFoundError("Nenhum SRT encontrado pra nenhum idioma")

if IDIOMA_MESTRE not in legendas_por_idioma_raw:
    print(f"⚠️  O idioma mestre ('{IDIOMA_MESTRE}') não foi encontrado — as legendas dos outros "
          f"idiomas estão sincronizadas em relação a ele, então isso é só um aviso informativo, "
          f"não impede o resto de rodar.")


  ✅ PT: 67 bloco(s)
  ✅ EN: 43 bloco(s)
  ✅ ES: 37 bloco(s)
  ✅ FR: 41 bloco(s)
  ✅ KO: 49 bloco(s)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  4. ANALISAR E CLASSIFICAR                                       ║
# ║  Stanza pra PT/EN/ES/FR, Kiwi pro KO. Três camadas:              ║
# ║    bruto (a origem)  →  regras (a central)  →  sua correção      ║
# ╚══════════════════════════════════════════════════════════════════╝
import stanza
from pathlib import Path

import analise
from renderizacao import (salvar_classificacao_multicolor,
                          carregar_classificacao_multicolor, classificacao_confere)
from revisao_classes import carregar_central, corrigir_pontuacao, remapear

# A CENTRAL DE CORREÇÕES AUTOMÁTICAS. Uma regra malformada é recusada aqui,
# na leitura -- não na hora de aplicar, quando ela simplesmente nunca casaria
# e ninguém descobriria que a correção parou de acontecer.
CENTRAL = carregar_central()
print(f"📖 Central de correções: {len(CENTRAL)} regra(s)")
for _r in CENTRAL:
    print(f"      {_r['id']:28} {_r['idiomas']}")
print()


def _do_drive(nome):
    """Baixa do Drive e devolve o caminho local, ou None se não existir."""
    local = Path(nome)
    return local if drive_client.download(config.pasta_oracao, nome, local) and local.exists() else None


def _bruto_reaproveitavel(idioma):
    """O bruto salvo serve pra este SRT? Devolve (blocos, motivo_de_recusar).

    Reaproveitar o bruto é o que evita rodar o analisador de novo. Mas ele é
    da versão do SRT que o gerou: se a legenda mudou, as peças carregam o
    TEXTO antigo e o vídeo sairia exibindo as palavras de antes, com o SRT
    novo parado ao lado. Nada falha; sai errado.
    """
    local = _do_drive(config.nome_analise_bruta(idioma))
    if local is None:
        return None, "não está no Drive"
    try:
        blocos, cab = analise.carregar(local)
    except ValueError as e:
        return None, str(e)
    divergencia = analise.confere(blocos, legendas_por_idioma_raw[idioma])
    if divergencia:
        return None, f"é de outra versão da legenda ({divergencia})"
    return (blocos, cab), None


def _analisar(idioma):
    """Roda o analisador e devolve os blocos brutos + (analisador, versão)."""
    legendas = legendas_por_idioma_raw[idioma]
    if idioma == IDIOMA_KIWI:
        blocos = [analise.de_kiwi(kiwi.analyze(leg.texto)[0][0], leg.texto,
                                  leg.inicio_ms, leg.fim_ms)
                  for leg in legendas]
        import kiwipiepy
        return blocos, "kiwi", getattr(kiwipiepy, "__version__", "?")
    codigo = IDIOMAS_STANZA[idioma]
    stanza.download(codigo, verbose=False)
    nlp = stanza.Pipeline(codigo, processors="tokenize,pos,lemma", verbose=False)
    blocos = [analise.de_stanza(nlp(leg.texto), leg.texto, leg.inicio_ms, leg.fim_ms)
              for leg in legendas]
    return blocos, "stanza", getattr(stanza, "__version__", "?")


blocos_por_idioma = {}
for idioma in list(IDIOMAS_STANZA.keys()) + [IDIOMA_KIWI]:
    if idioma not in legendas_por_idioma_raw:
        continue
    print(f"── {idioma.upper()} " + "─" * 50)

    # ── 1. O BRUTO: reaproveita, ou roda o analisador ──────────────────────
    guardado, motivo = _bruto_reaproveitavel(idioma)
    if guardado:
        brutos, cabecalho = guardado
        analisador = cabecalho["analisador"]
        print(f"   ⚡ bruto reaproveitado ({analisador} {cabecalho['versao_analisador']}, "
              f"{cabecalho['gerado_em']}) — sem rodar o analisador")
    else:
        print(f"   🔬 analisando ({motivo})")
        brutos, analisador, versao = _analisar(idioma)
        nome = config.nome_analise_bruta(idioma)
        analise.salvar(brutos, Path(nome), idioma=idioma,
                       analisador=analisador, versao=versao)
        drive_client.upload(Path(nome), config.pasta_oracao, "application/json")
        print(f"   💾 bruto salvo: {nome} ({analisador} {versao})")

    # ── 2. A CLASSIFICAÇÃO ANTERIOR, se houver: é dela que vem sua correção ─
    salvos = None
    local_classes = _do_drive(config.nome_classificacao_multicolor(idioma))
    if local_classes is not None:
        candidatos = carregar_classificacao_multicolor(local_classes)
        if classificacao_confere(candidatos, legendas_por_idioma_raw[idioma]):
            print(f"   ⚠️  a classificação salva é de outra versão da legenda — descartada")
        else:
            salvos = candidatos

    # ── 3. MAPEAR + CENTRAL + preservar o que a mão mexeu ──────────────────
    blocos, relatorio = remapear(salvos, brutos, idioma, analisador, CENTRAL)
    blocos, mud_pontuacao = corrigir_pontuacao(blocos)
    for _linha in relatorio + mud_pontuacao:
        print(f"      {_linha}")

    _div = classificacao_confere(blocos, legendas_por_idioma_raw[idioma])
    if _div:
        print(f"   🚩 a classificação NÃO reproduz o SRT — {_div}.")
        print(f"      Não use sem olhar; o texto na tela sai diferente da legenda.")

    blocos_por_idioma[idioma] = blocos
    nome = config.nome_classificacao_multicolor(idioma)
    salvar_classificacao_multicolor(blocos, Path(nome))
    drive_client.upload(Path(nome), config.pasta_oracao, "application/json")
    print(f"   ✅ {len(blocos)} bloco(s), "
          f"{sum(len(b['pecas']) for b in blocos)} peças → {nome}")

print()
print("✅ Siga pra célula 5 (revisar)")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  5. REVISAR — a lista de suspeitas, a planilha e a página        ║
# ║  Nada é queimado aqui: esta célula só te mostra o que olhar.     ║
# ╚══════════════════════════════════════════════════════════════════╝
from revisao_classes import suspeitas, exportar_csv, pagina_revisao

# Por que uma lista de suspeitas em vez de "confira tudo": são ~3.300 peças
# nos 5 idiomas. As três regras apontaram 39 delas no Mateus 2 (1,2%), o que
# dá pra ler em cinco minutos:
#
#   sem regra          o analisador devolveu uma etiqueta que o classificador
#                      não trata -- a cor saiu de um chute, não de uma análise
#   classe instável    a mesma palavra recebeu outra classe no resto da legenda
#   classe rara        classe com até 3 ocorrências no idioma inteiro
#
# NÃO é lista de erro: ambiguidade legítima entra junto ("où" é advérbio numa
# frase e pronome na outra). É lista de ONDE olhar.
_suspeitas = suspeitas(blocos_por_idioma)
print(f"🔎 {len(_suspeitas)} peça(s) pra conferir")
_por_idioma = {}
for _s in _suspeitas:
    _por_idioma.setdefault(_s.idioma, []).append(_s)
for _idioma, _lista in sorted(_por_idioma.items()):
    print(f"\n── {_idioma.upper()} ({len(_lista)}) ──")
    for _s in _lista:
        print(f"   bloco {_s.bloco:2d}  «{_s.palavra}» = {_s.classe}")
        print(f"              {_s.motivo}")

# ── Os dois arquivos da revisão ────────────────────────────────────────────
# A página é pra ACHAR o erro: mostra a legenda pintada com as cores de
# verdade, os 5 idiomas empilhados como no vídeo. Nome de classe não é o que
# se enxerga -- o erro aparece quando "Herodes" sai preto no meio de nomes
# próprios amarelos.
# O CSV é pra CORRIGIR: abre no Sheets, você muda a coluna `classe`, e a
# célula 6 lê de volta.
_csv = Path(config.nome_revisao_classes)
_html = Path(config.nome_pagina_revisao_classes)
exportar_csv(blocos_por_idioma, _csv, _suspeitas)
pagina_revisao(blocos_por_idioma, _html, _suspeitas, titulo=NOME_ORACAO)
drive_client.upload(_csv, config.pasta_oracao, "text/csv")
drive_client.upload(_html, config.pasta_oracao, "text/html")
print(f"\n💾 {_csv.name} e {_html.name} — no Drive e aqui embaixo pra baixar")

from google.colab import files
files.download(str(_html))
files.download(str(_csv))
print("\n➡️  Abra o HTML no navegador. Se estiver tudo certo, PULE a célula 6")
print("    e vá direto pra 7 (gerar o .ass).")

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  6. APLICAR A CORREÇÃO MANUAL — só se você mexeu no CSV          ║
# ║  Nada mudou na revisão? Pule esta célula.                        ║
# ╚══════════════════════════════════════════════════════════════════╝
from revisao_classes import aplicar_csv, sugerir_regras, ErroDeRevisao
from google.colab import files

print("Envie o CSV corrigido (ou cancele pra pular):")
_enviados = files.upload()

if not _enviados:
    print("⏭️  Nada enviado — seguindo com a classificação como está.")
else:
    _nome_csv = list(_enviados.keys())[0]
    # Recusar é melhor que aceitar torto: uma classe com erro de digitação
    # sairia CINZA no vídeo, igualzinho a uma pontuação, e ninguém veria
    # antes de assistir.
    try:
        _corrigido, _mudancas = aplicar_csv(blocos_por_idioma, Path(_nome_csv))
    except ErroDeRevisao as _e:
        print(f"❌ {_e}")
        raise SystemExit("Corrija o CSV e rode esta célula de novo.")

    print(f"\n✏️  {len(_mudancas)} correção(ões):")
    for _m in _mudancas:
        print(f"   {_m['idioma']} bloco {_m['bloco']:2d}  «{_m['palavra']}»  "
              f"{_m['de']} → {_m['para']}")

    # `classe` muda; `classe_automatica` NÃO -- é a diferença entre as duas
    # que marca "isto aqui foi a mão", e é ela que faz a correção sobreviver
    # quando a regra mudar e a classificação for refeita do bruto.
    for _idioma in blocos_por_idioma:
        blocos_por_idioma[_idioma] = _corrigido[_idioma]
        _nome = config.nome_classificacao_multicolor(_idioma)
        salvar_classificacao_multicolor(_corrigido[_idioma], Path(_nome))
        drive_client.upload(Path(_nome), config.pasta_oracao, "application/json")
    print("\n💾 Classificação corrigida salva no Drive.")

    # ── Pra correção não morrer com este vídeo ─────────────────────────────
    # O que você corrigiu MAIS DE UMA VEZ provavelmente é defeito do
    # analisador, não do contexto -- e defeito do analisador vai se repetir em
    # Mateus 3, 4, 5. Vira regra na central e nunca mais aparece.
    _sugestao = sugerir_regras(_mudancas)
    if _sugestao:
        print("\n" + "═" * 60)
        print("📌 Correções que se REPETIRAM — candidatas a virar regra.")
        print("   Escreva o 'porque' e acrescente em")
        print("   pipeline/dados_lexico/classes-correcoes.json (no repositório,")
        print("   não no Drive: o sincronizador sobrescreve o do Drive).")
        print("   Sem o 'porque' a central recusa a regra na leitura.")
        print("═" * 60)
        print(_sugestao)

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  7. GERAR O .ASS COM CAIXA COLORIDA                              ║
# ╚══════════════════════════════════════════════════════════════════╝
from renderizacao import gerar_ass

config_render = PipelineConfig(NOME_ORACAO=NOME_ORACAO, PASTA_DRIVE_RAIZ=PASTA_DRIVE_RAIZ, IDIOMA_MESTRE=IDIOMA_MESTRE)

caminho_ass = gerar_ass(blocos_por_idioma, config_render, box_border=BOX_BORDER,
                        espaco_colado=ESPACO_ENTRE_PECAS_COLADAS,
                        margem_lateral=MARGEM_LATERAL,
                        mostrar_siglas=CAMADAS_ATIVAS["siglas_idioma"])
print(f"✅ Legenda gerada: {caminho_ass}")
print("\nRode a célula 8 pra baixar e revisar. Só rode a célula 9 (salvar no Drive) depois de confirmar que ficou bom.")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  8. BAIXAR O .ASS (pra revisar / corrigir manualmente)           ║
# ╚══════════════════════════════════════════════════════════════════╝
from google.colab import files
files.download(str(caminho_ass))


In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  9. SALVAR NO DRIVE — só rode depois de conferir que ficou bom   ║
# ║  (revise o .ass baixado na célula 8 antes de rodar essa aqui)    ║
# ╚══════════════════════════════════════════════════════════════════╝
caminho_no_drive = drive_client.upload(caminho_ass, config_render.pasta_oracao)
print(f"✅ Salvo no Drive: {caminho_no_drive}")
